In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 确保能导入当前目录下的 sim 文件夹
from sim import ProfileRegistry, GMMCellEngine, NBCellEngine, DropletFactory

# 设置随机种子保证可重复性
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# 物理实验参数
H = 16                   # HTO 标签总数
N_cells = 30000          # 拟生成的总细胞数
M_droplets = 100000      # 拟生成的总液滴数（槽位）

# 统计分布参数
# 可选: "GMM" (连续空间, 模拟CLR) 或 "NB" (离散计数, 模拟Raw Counts)
DIST_TYPE = "NB" 

# 信号强度设置
MU_HIGH = 7.0            # 阳性信号均值
MU_LOW = 1.0             # 阴性背景均值
AMBIENT_LEVEL = 0.15     # 环境背景水平 (0.1 - 0.3 较为真实)

In [ ]:
# 初始化配置
reg = ProfileRegistry(
    h_tags=H, 
    mu_high=MU_HIGH, 
    mu_low=MU_LOW, 
    ambient_level=AMBIENT_LEVEL, 
    seed=SEED
)

print(f"成功初始化 {len(reg.htos)} 个 HTO 通道配置。")
print(f"全局环境背景向量 (前5个): {reg.ambient_vector[:5]}")

In [ ]:
# 引擎选择逻辑
if DIST_TYPE == "GMM":
    engine = GMMCellEngine(reg)
else:
    engine = NBCellEngine(reg)

# 生成原始细胞数据
cell_matrix, donor_ids = engine.generate(n_cells=N_cells)

print(f"细胞生成完成: 矩阵形状为 {cell_matrix.shape}, 数据类型为 {cell_matrix.dtype}")

In [ ]:
# 初始化工厂
factory = DropletFactory(m_droplets=M_droplets)

# 聚合、生成真值、去除空液滴
final_matrix, ground_truth = factory.produce(
    cell_matrix, 
    donor_ids, 
    reg.htos
)

# 显示统计报告 (包含 Singlet, MSM, SSM 占比)
factory.display_report(ground_truth)

In [ ]:
# 绘制第一个 HTO 通道的分布图
sample_hto_idx = 0
data_to_plot = final_matrix[:, sample_hto_idx].numpy()

plt.figure(figsize=(10, 4))
if DIST_TYPE == "NB":
    sns.histplot(np.log1p(data_to_plot), bins=50, kde=True, color='skyblue')
    plt.xlabel("Log1p Counts")
else:
    sns.histplot(data_to_plot, bins=50, kde=True, color='salmon')
    plt.xlabel("Latent Value (GMM)")

plt.title(f"Distribution of {reg.htos[sample_hto_idx]} (Non-Empty Droplets)")
plt.show()

In [ ]:
# 创建数据目录
os.makedirs("data", exist_ok=True)

# 保存 Counts 矩阵 (如果是 NB 则保存为整数)
# 实际项目中建议保存为 .mtx 格式，这里先存为简洁的 torch 或 csv
torch.save(final_matrix, "data/hto_matrix.pt")

# 保存真值表
ground_truth.to_csv("data/ground_truth.csv", index=False)

print("模拟数据已成功保存至 data/ 文件夹。")